# 第2章 预备知识 · 2.2 数据预处理

真实数据常含缺失值。本笔记演示 d2l 标准流水线：造数据 → 读入 → 补数值缺失 → 独热编码 → 转张量。

In [12]:
import torch

## 1. 创建并写入带缺失值的数据文件

`os.makedirs` + `os.path.join` 跨平台建目录（路径里的 `..` 表示上级目录），再写一份故意埋了 `NA` 占位符的 CSV，模拟不干净的数据。

In [13]:
import os

os.makedirs(os.path.join('..', 'data'), exist_ok=True)
data_file = os.path.join('..', 'data', 'house_tiny.csv')
with open(data_file, 'w') as f:
    f.write('NumRooms,Alley,Price\n')  # 列名
    f.write('NA,Pave,127500\n')  # 每行表示一个数据样本
    f.write('2,NA,106000\n')
    f.write('4,NA,178100\n')
    f.write('NA,NA,140000\n')

## 2. 读取 CSV 并查看

`pd.read_csv` 把文件读成 DataFrame；文件里的 `NA` 会被自动识别成真正的缺失值 `NaN`。

In [14]:
# 如果没有安装pandas，只需取消对以下行的注释来安装pandas
# !pip install pandas
import pandas as pd

data = pd.read_csv(data_file)
print(data)

   NumRooms Alley   Price
0       NaN  Pave  127500
1       2.0   NaN  106000
2       4.0   NaN  178100
3       NaN   NaN  140000


## 3. 处理数值型缺失值

用 `iloc[:, 0:2]` 按列位置拆出特征 `inputs`（前两列）和标签 `outputs`（价格）；`fillna(mean)` 用每列均值填补缺失。**pandas 2.x 需加 `numeric_only=True`**，否则对字符串列求均值会报错。

In [15]:
inputs, outputs = data.iloc[:, 0:2], data.iloc[:, 2]
inputs = inputs.fillna(inputs.mean(numeric_only=True))  # 用均值填充缺失值
print(inputs)

   NumRooms Alley
0       3.0  Pave
1       2.0   NaN
2       4.0   NaN
3       3.0   NaN


## 4. 处理类别型缺失值（独热编码）

`pd.get_dummies(dummy_na=True)` 把 `Alley` 这类文本列变成 0/1 哑变量，缺失也单独占一列。

In [16]:
inputs = pd.get_dummies(inputs, dummy_na=True)
print(inputs)

   NumRooms  Alley_Pave  Alley_nan
0       3.0        True      False
1       2.0       False       True
2       4.0       False       True
3       3.0       False       True


## 5. 转换成张量

`inputs.to_numpy(dtype=float)` 先转成 NumPy，再 `torch.tensor` 变成模型能吃的张量。

In [17]:
X = torch.tensor(inputs.to_numpy(dtype=float))
y = torch.tensor(outputs.to_numpy(dtype=float))
X, y

(tensor([[3., 1., 0.],
         [2., 0., 1.],
         [4., 0., 1.],
         [3., 0., 1.]], dtype=torch.float64),
 tensor([127500., 106000., 178100., 140000.], dtype=torch.float64))